In [30]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import os
import json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

In [63]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")

# ID mappings (from 02_id_encoder.ipynb)
id_mappings_dir = os.path.join(preprocessed_dir, "id_mappings")
user_id_2idx_path = os.path.join(id_mappings_dir, "user_id_2idx.json")
item_id_2idx_path = os.path.join(id_mappings_dir, "item_id_2idx.json")
scene_2idx_path = os.path.join(id_mappings_dir, "scene_2idx.json")
id_vocab_config_path = os.path.join(id_mappings_dir, "id_vocab_config.json")

# Precomputed embeddings
text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")               # (N_items, 768)
image_embeddings_aligned_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")              # (N_interactions, D_time)
time_feature_cols_path = os.path.join(preprocessed_dir, "time_features_columns.json")      # list of column names

# fusion artifacts from this notebook will be saved here
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")
os.makedirs(fusion_artifacts_dir, exist_ok=True)

In [65]:
# sanity check
print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("id_mappings_dir:", id_mappings_dir)
print("text_embeddings_path:", text_embeddings_path)
print("image_embeddings_path:", image_embeddings_aligned_path)
print("time_features_path:", time_features_path)
print("fusion_artifacts_dir:", fusion_artifacts_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
id_mappings_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/id_mappings
text_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/text_embeddings.npy
image_embeddings_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base_aligned.pt
time_features_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_features_final.pt
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts


In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [68]:
# Next, we will load Embedding Sizes (Dynamic Sizing)
# Load shapes without pulling everything into GPU memory
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
text_dim = text_embeddings.shape[1]

image_embeddings = torch.load(image_embeddings_aligned_path, map_location="cpu")
image_dim = image_embeddings.shape[1]

time_features = torch.load(time_features_path, map_location="cpu")
time_raw_dim = time_features.shape[1]

In [76]:
# Load time feature names
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

print("--- Embedding/Feature Dimensions ---")
print("Text embedding dim:", text_dim)
print("Image embedding dim:", image_dim)
print("Time raw feature dim:", time_raw_dim)
print("Time feature columns:", time_feature_cols)

--- Embedding/Feature Dimensions ---
Text embedding dim: 1536
Image embedding dim: 768
Time raw feature dim: 5
Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [70]:
# Load vocab sizes (for ID embeddings)
with open(id_vocab_config_path, "r") as f:
    id_vocab_config = json.load(f)

user_vocab_size = int(id_vocab_config["num_users"])
item_vocab_size = int(id_vocab_config["num_items"])
scene_vocab_size = int(id_vocab_config["num_scenes"])

print("\n--- ID Vocabulary Sizes ---")
print("user_vocab_size:", user_vocab_size)
print("item_vocab_size:", item_vocab_size)
print("scene_vocab_size:", scene_vocab_size)


--- ID Vocabulary Sizes ---
user_vocab_size: 67615
item_vocab_size: 59142
scene_vocab_size: 5


## Load All Precomputed Embeddings / Feature Shapes

In this step, we validate and summarize the shapes of all modality inputs that will enter the ColdFusionX fusion module.
These inputs reference back to earlier notebooks:
*   Text embeddings from 03_text_encoder_BERT.ipynb
*   Image embeddings from 04_image_encoder.ipynb (ViT-base CLS, 768-d)
*   Time features from 05_time_encoder.ipynb (raw time vector, 5-d)
*   ID vocab sizes from 02_id_encoder.ipynb (user/item/scene mappings)

To keep memory usage low, we first load arrays/tensors in a lightweight way (e.g., NumPy memmap, CPU tensors).
We then run sanity checks to ensure dimensions match expected vocab sizes and that indexing will remain consistent during model training.

In [71]:
# Lightweight load
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")  # interaction-level
image_embeddings = torch.load(image_embeddings_aligned_path, map_location="cpu")  # item-level aligned
time_features = torch.load(time_features_path, map_location="cpu")  # interaction-level

num_text_rows, text_dim = text_embeddings.shape
num_image_rows, image_dim = image_embeddings.shape
num_time_rows, time_raw_dim = time_features.shape

In [72]:
print("- Precomputed Embeddings / Feature Shapes ")
print(f"Text embeddings:  shape = ({num_text_rows}, {text_dim})  [interaction-level]")
print(f"Image embeddings: shape = ({num_image_rows}, {image_dim}) [item-level aligned]")
print(f"Time features:    shape = ({num_time_rows}, {time_raw_dim}) [interaction-level]")

------ Precomputed Embeddings / Feature Shapes ------
Text embeddings:  shape = (3730329, 1536)  [interaction-level]
Image embeddings: shape = (59142, 768) [item-level aligned]
Time features:    shape = (3730329, 5) [interaction-level]


In [73]:
# Load time feature names
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

print("\nTime feature columns:", time_feature_cols)


Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [75]:
# Load missing mask
image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")
print("Missing image embeddings (count):", image_missing_mask.sum().item())

Missing image embeddings (count): 2


In [77]:
# item-level alignment (image embeddings)
assert num_image_rows == item_vocab_size, (
    f"Mismatch: image_embeddings rows ({num_image_rows}) != item_vocab_size ({item_vocab_size})"
)

# interaction-level alignment (text embeddings + time features)
assert num_text_rows == num_time_rows, (
    f"Mismatch: text rows ({num_text_rows}) != time rows ({num_time_rows})"
)

# time feature column alignment
assert time_raw_dim == len(time_feature_cols), (
    f"Mismatch: time_raw_dim ({time_raw_dim}) != number of time feature columns ({len(time_feature_cols)})"
)

In [78]:
# Basic numerical checks
# Sample small slices to avoid scanning all 3.7M rows
sample_idx = np.random.randint(0, num_time_rows, size=2048)

text_sample = np.array(text_embeddings[sample_idx])
time_sample = time_features[sample_idx].numpy()
image_sample = image_embeddings[np.random.randint(0, num_image_rows, size=256)].numpy()

assert not np.isnan(text_sample).any(), "NaNs found in text embeddings sample."
assert not np.isnan(time_sample).any(), "NaNs found in time features sample."
assert not np.isnan(image_sample).any(), "NaNs found in image embeddings sample."

print("Sanity checks passed (item-level + interaction-level alignment confirmed).")

Sanity checks passed (item-level + interaction-level alignment confirmed).


In [80]:
!git add .